<a href="https://colab.research.google.com/github/Aswini-Dileep/AI-ML-Project/blob/main/ai_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Cell 1: Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score

print("All libraries imported successfully!")


All libraries imported successfully!


In [ ]:

# Cell 2: Load dataset

df = pd.read_csv("/content/synthetic_indian_school_dataset_1000.csv")

print("Dataset loaded successfully!")


FileNotFoundError: [Errno 2] No such file or directory: '/content/synthetic_indian_school_dataset_1000.csv'

In [ ]:

# Cell 3: Display first five rows

df.head()


NameError: name 'df' is not defined

In [ ]:

# Cell 4: Check number of rows and columns

print("Number of observations:", df.shape[0])
print("Number of features:", df.shape[1])


In [ ]:

# Cell 5: Dataset information

df.info()


In [ ]:

# Cell 6: Statistical summary

df.describe()


In [ ]:

# Cell 7: Check missing values

print("Missing values in each column:")
print(df.isnull().sum())


In [ ]:

# Cell 8: Check duplicate rows

print("Number of duplicate rows:", df.duplicated().sum())


In [ ]:

# Cell 9: Check duplicate school IDs

print("Duplicate School IDs:", df["School_ID"].duplicated().sum())


In [ ]:

# Cell 10: Select numerical columns

numerical_df = df.select_dtypes(include=["int64", "float64"])

print("Numerical columns:")
print(numerical_df.columns.tolist())


In [ ]:

# Cell 11: Plot numerical feature distributions

numerical_df.hist(figsize=(15, 10), bins=20)

plt.suptitle("Distribution of Numerical Features")
plt.tight_layout()
plt.show()


In [ ]:

# Cell 12: Select clustering features

features = [
    "Students",
    "Teachers",
    "Study_Groups",
    "Subjects"
]

X = df[features]

print("Selected features:")
print(X.head())

print("\nDataset shape:", X.shape)


In [ ]:

# Cell 13: Apply Min-Max Normalization

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for easy viewing
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=features
)

print(X_scaled_df.head())


In [ ]:

# Cell 14: Check minimum and maximum values

print("Minimum values:")
print(X_scaled_df.min())

print("\nMaximum values:")
print(X_scaled_df.max())


In [ ]:

# Experiment 2: Standard K-Means from scratch

def standard_kmeans(X, k, max_iterations=100, tolerance=0.0001):

    # Step 1: Initialize random centroids
    np.random.seed(42)

    random_indices = np.random.choice(
        len(X), k, replace=False
    )

    centroids = X[random_indices]

    for iteration in range(max_iterations):

        # Step 2: Calculate distances
        distances = np.linalg.norm(
            X[:, np.newaxis] - centroids,
            axis=2
        )

        # Step 3: Assign each observation to nearest centroid
        labels = np.argmin(distances, axis=1)

        # Step 4: Update centroids
        new_centroids = []

        for cluster in range(k):

            cluster_points = X[labels == cluster]

            if len(cluster_points) > 0:
                new_centroid = cluster_points.mean(axis=0)
            else:
                new_centroid = centroids[cluster]

            new_centroids.append(new_centroid)

        new_centroids = np.array(new_centroids)

        # Step 5: Check convergence
        difference = np.linalg.norm(
            new_centroids - centroids
        )

        centroids = new_centroids

        if difference < tolerance:
            break

    return labels, centroids, iteration + 1


In [ ]:

# Run Standard K-Means with K = 3

X_array = X_scaled_df.to_numpy()

labels, centroids, iterations = standard_kmeans(
    X_array,
    k=3
)

print("K-Means completed successfully!")
print("Number of clusters:", 3)
print("Number of iterations:", iterations)

print("\nCentroids:")
print(centroids)


In [ ]:

# Cell 15: Visualize K-Means clusters

plt.figure(figsize=(10, 6))

plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=labels,
    cmap="viridis",
    s=30
)

plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    color="red",
    marker="X",
    s=250,
    label="Centroids"
)

plt.xlabel("Students (Scaled)")
plt.ylabel("Teachers (Scaled)")
plt.title("Standard K-Means Clustering (K=3)")
plt.legend()
plt.show()


In [ ]:

# Cell 16: Calculate SSE

sse = 0

for i in range(len(X_array)):

    assigned_centroid = centroids[labels[i]]

    distance = np.linalg.norm(
        X_array[i] - assigned_centroid
    )

    sse += distance ** 2

print("Sum of Squared Errors (SSE):", sse)


In [ ]:

# Cell 17: Calculate Silhouette Score

score = silhouette_score(X_scaled, labels)

print("Silhouette Score:", score)


In [ ]:

# Cell 18: Store baseline experiment results

baseline_results = pd.DataFrame({
    "Method": ["Random Initialization"],
    "K": [3],
    "Iterations": [iterations],
    "SSE": [sse],
    "Silhouette_Score": [score]
})

baseline_results


In [ ]:

# Cell 19: Standard K-Means using Scikit-learn

from sklearn.cluster import KMeans

sklearn_kmeans = KMeans(
    n_clusters=3,
    init="random",
    n_init=1,
    max_iter=100,
    random_state=42,
    tol=0.0001
)

sklearn_labels = sklearn_kmeans.fit_predict(X_scaled)

sklearn_centroids = sklearn_kmeans.cluster_centers_

print("Scikit-learn K-Means")
print("--------------------")

print("Iterations:", sklearn_kmeans.n_iter_)
print("SSE:", sklearn_kmeans.inertia_)

print(
    "Silhouette Score:",
    silhouette_score(X_scaled, sklearn_labels)
)


In [ ]:

# Cell 20: Elbow Method

sse_values = []
k_values = range(2, 11)

for k in k_values:

    labels, centroids, iterations = standard_kmeans(
        X_array,
        k=k
    )

    sse = 0

    for i in range(len(X_array)):
        cluster = labels[i]
        sse += np.sum((X_array[i] - centroids[cluster]) ** 2)

    sse_values.append(sse)

    print(f"K = {k}, SSE = {sse:.4f}")


NameError: name 'standard_kmeans' is not defined

In [ ]:

# Cell 21: Elbow graph

plt.figure(figsize=(8,5))

plt.plot(list(k_values), sse_values, marker='o')

plt.xlabel("Number of Clusters (K)")
plt.ylabel("SSE")
plt.title("Elbow Method")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


In [ ]:

# Cell 22: SSE table

elbow_results = pd.DataFrame({
    "K": list(k_values),
    "SSE": sse_values
})

elbow_results
